In [15]:
import numpy as np
import pandas as pd
import matplotlib as plt 

In [16]:
data = pd.read_csv('/kaggle/input/competitions/digit-recognizer/train.csv')

In [17]:
data.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [18]:
data = np.array(data) # working  with linear algebra so conversion is necessary 
m,n = data.shape
np.random.shuffle(data) # Shuffling to avoid overfitting

data_dev = data[0:1000].T
Y_dev = data_dev[0]
X_dev = data_dev[1:n] / 255.

data_train = data[1000 : m].T
Y_train = data_train[0]
X_train = data_train[1:n] / 255.

In [19]:
def init_params():
    W1 = np.random.rand(10, 784) - 0.5
    b1 = np.random.rand(10, 1) - 0.5
    W2 = np.random.rand(10, 10) - 0.5
    b2 = np.random.rand(10, 1) - 0.5
    return W1, b1, W2, b2

def ReLU(Z):
    return np.maximum(Z, 0)

def softmax(Z):
    Z = Z - np.max(Z, axis=0, keepdims=True)
    exp_Z = np.exp(Z)
    A = exp_Z / np.sum(exp_Z, axis=0, keepdims=True)
    return A
    
def forward_prop(W1, b1, W2, b2, X):
    Z1 = W1.dot(X) + b1
    A1 = ReLU(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2

def ReLU_deriv(Z):
    return Z > 0

def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, Y.max() + 1))
    one_hot_Y[np.arange(Y.size), Y] = 1
    one_hot_Y = one_hot_Y.T
    return one_hot_Y

def deriv_ReLU(Z):
    return Z > 0


def backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y):
    m = Y.size
    one_hot_Y = one_hot(Y)
    dZ2 = A2 - one_hot_Y
    dW2 = 1 / m * dZ2.dot(A1.T)
    db2 = 1 / m * np.sum(dZ2, axis=1, keepdims=True)
    dZ1 = W2.T.dot(dZ2) * ReLU_deriv(Z1)
    dW1 = 1 / m * dZ1.dot(X.T)
    db1 = 1 / m * np.sum(dZ1, axis=1, keepdims=True)
    return dW1, db1, dW2, db2

def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1    
    W2 = W2 - alpha * dW2  
    b2 = b2 - alpha * db2    
    return W1, b1, W2, b2




In [20]:
def get_predictions(A2):
    return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
    print(predictions, Y)
    return np.sum(predictions == Y) / Y.size

def gradient_descent(X, Y, alpha, iterations):
    W1, b1, W2, b2 = init_params()
    for i in range(iterations):
        Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, X)
        dW1, db1, dW2, db2 = backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)
        if i % 10 == 0:
            print("Iteration: ", i)
            predictions = get_predictions(A2)
            print(get_accuracy(predictions, Y))
    return W1, b1, W2, b2

In [21]:
W1, b1, W2, b2 = gradient_descent(X_train, Y_train, 0.10, 500)

Iteration:  0
[9 9 9 ... 6 5 9] [4 4 6 ... 8 1 8]
0.08495121951219513
Iteration:  10
[4 6 0 ... 4 3 7] [4 4 6 ... 8 1 8]
0.18326829268292683
Iteration:  20
[4 4 6 ... 4 1 7] [4 4 6 ... 8 1 8]
0.30565853658536585
Iteration:  30
[4 4 6 ... 4 1 7] [4 4 6 ... 8 1 8]
0.4185609756097561
Iteration:  40
[4 4 6 ... 1 1 7] [4 4 6 ... 8 1 8]
0.49290243902439024
Iteration:  50
[4 4 6 ... 8 1 7] [4 4 6 ... 8 1 8]
0.5483170731707318
Iteration:  60
[4 4 6 ... 8 1 9] [4 4 6 ... 8 1 8]
0.5930243902439024
Iteration:  70
[4 4 6 ... 8 1 9] [4 4 6 ... 8 1 8]
0.6286341463414634
Iteration:  80
[4 4 6 ... 8 1 9] [4 4 6 ... 8 1 8]
0.6564146341463415
Iteration:  90
[4 4 6 ... 8 1 9] [4 4 6 ... 8 1 8]
0.6797317073170732
Iteration:  100
[4 4 6 ... 8 1 9] [4 4 6 ... 8 1 8]
0.697
Iteration:  110
[4 4 6 ... 8 1 9] [4 4 6 ... 8 1 8]
0.7129024390243902
Iteration:  120
[4 4 6 ... 8 1 9] [4 4 6 ... 8 1 8]
0.7250487804878049
Iteration:  130
[4 4 6 ... 8 1 9] [4 4 6 ... 8 1 8]
0.7362439024390244
Iteration:  140
[4 4 6 ...